In [ ]:
from leandojo import lean_dojo
repo = lean_dojo.clone(
    "https://github.com/leanprover-community/mathlib4",
    commit_hash="main"
)
for theorem in repo.theorems:
    print(theorem.name, theorem.statement, theorem.proof)

In [ ]:
from leandojo import lean_dojo
from openai import OpenAI
client = OpenAI()
state = lean_dojo.get_initial_state("theorem_name")
while not state.is_done():
    prompt = f"Current proof state:\n{state}\nGenerate next tactic:"
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    tactic = response.choices[0].message.content
    state = lean_dojo.apply_tactic(state, tactic)
    if state.has_error():
        # 失败信息回传给模型重试
        prompt = f"Error: {state.error}\nRetry with different tactic:"

In [1]:
import random

N = 32            # 搜索域 {0, ..., N-1}
SEED = 1          # 固定随机种子，保证输出可复现
B = 2000          # 基线阶段每个目标规模的提议预算
K_LIST = (4, 5, 6, 7)
K_TARGET = 7      # 迭代搜索的目标规模（计数上界允许 8）

rng = random.Random(SEED)


# ---------- 确定性验证器：精确判定，无随机性、无近似 ----------
def is_sidon(s):
    """判定 s 是否为 Sidon 集。返回 (是否通过, 冲突描述)。"""
    seen = {}
    for i in range(len(s)):
        for j in range(i + 1, len(s)):
            diff = s[j] - s[i]
            if diff in seen:
                return False, (seen[diff], (s[i], s[j]), diff)
            seen[diff] = (s[i], s[j])
    return True, None


def counting_max(n):
    """计数上界：k 个元素产生 C(k,2) 个互异正差，须全部落在 1..n-1。"""
    k = 1
    while (k + 1) * k // 2 <= n - 1:
        k += 1
    return k


# ---------- 生成器模式 A：无引导的均匀提议 ----------
def propose_uniform(k):
    return sorted(rng.sample(range(N), k))


# ---------- 生成器模式 B：贪心构造（边构造边做增量检查） ----------
def greedy_fill(seed_elems, pool):
    """从 seed_elems 出发，按随机顺序贪心加入不破坏 Sidon 性质的元素。"""
    s = sorted(seed_elems)
    diffs = set(s[j] - s[i] for i in range(len(s)) for j in range(i + 1, len(s)))
    order = pool[:]
    rng.shuffle(order)
    for x in order:
        # 注意一：必须用绝对差。初版此处写作 x - y（有符号），
        # 与验证器按升序集合计算的无符号差不一致，导致其候选被验证器整体拦截。
        # 注意二：当 x 恰为两个已有点的中点时，两条新差的绝对值会相同，
        # 因此除查重于已有差集外，还须检查新差内部互不相同（此缺陷同样被验证器拦截过）。
        new_diffs = [abs(x - y) for y in s]
        if len(set(new_diffs)) != len(new_diffs):
            continue
        if set(new_diffs) & diffs:
            continue
        diffs.update(new_diffs)
        s.append(x)
    return sorted(s)


def mutate(elite, keep_prob=0.6):
    """以精英为起点做随机变异：保留部分元素，再随机贪心补齐。"""
    keep = [x for x in elite if rng.random() < keep_prob]
    pool = [x for x in range(N) if x not in keep]
    return greedy_fill(keep, pool)


# ==================== 阶段 0：验证器自检 ====================
print("=== 阶段 0：验证器自检 ===")
ok1, _ = is_sidon([0, 1, 4, 10, 12, 25])
ok2, conflict2 = is_sidon([0, 1, 2, 4, 10, 20])
print("is_sidon([0, 1, 4, 10, 12, 25]) =", ok1)
print("is_sidon([0, 1, 2, 4, 10, 20]) =", ok2, "冲突：差 %d 重复，出现于 %s 与 %s"
      % (conflict2[2], conflict2[0], conflict2[1]))
print("计数上界（C(k,2) <= N-1）：k_max =", counting_max(N))

# ==================== 阶段 1：无引导提议（基线） ====================
print()
print("=== 阶段 1：无引导提议（均匀提议分布，每个目标规模 B=%d 次） ===" % B)
reject_log = []
for k in K_LIST:
    hits = 0
    logged = False
    for _ in range(B):
        cand = propose_uniform(k)
        ok, conflict = is_sidon(cand)
        if ok:
            hits += 1
        elif not logged:
            reject_log.append((k, cand, conflict))
            logged = True
    phat = hits / B
    if hits > 0:
        print("k=%d：提议 %d 次，通过 %d 次，经验命中率 p_hat = %.4f" % (k, B, hits, phat))
    else:
        print("k=%d：提议 %d 次，通过 0 次（p_hat < %.5f）" % (k, B, 1.0 / B))
print("被拒绝候选示例（验证器拦截）：")
for k, cand, conflict in reject_log:
    print("  k=%d 候选 %s：差 %d 重复，出现于 %s 与 %s"
          % (k, cand, conflict[2], conflict[0], conflict[1]))

# ==================== 阶段 2：被验证信号引导的迭代搜索 ====================
print()
print("=== 阶段 2：迭代搜索（贪心构造 + 变异 + 精英保留，目标规模 %d） ===" % K_TARGET)
elite = None
history = []
n_prop = 0
n_verif_reject = 0
n_select_reject = 0
for it in range(1, 3001):
    if elite is None:
        cand = greedy_fill([], list(range(N)))
    elif rng.random() < 0.2:
        cand = greedy_fill([], list(range(N)))     # 少量完全重启
    else:
        cand = mutate(elite)
    n_prop += 1
    ok, _ = is_sidon(cand)                          # 终验：整体再过一遍验证器
    if not ok:
        n_verif_reject += 1                         # 构造性生成下不应发生，作为监控
        continue
    if elite is None or len(cand) > len(elite):     # 保留规则：严格优于精英才接受
        elite = cand
        history.append((n_prop, len(cand), list(cand)))
        if len(elite) >= K_TARGET:
            break
    else:
        n_select_reject += 1

print("精英更新轨迹（提议序号, 精英规模, 精英集合）：")
for t, size, s in history:
    print("  第 %4d 次提议：规模 %d，精英 = %s" % (t, size, s))
print("总计提议 %d 次；验证器拒绝 %d 次；通过验证但未优于精英 %d 次（保留规则拒绝）"
      % (n_prop, n_verif_reject, n_select_reject))
print("最终精英（规模 %d）：%s" % (len(elite), elite))

# ==================== 阶段 3：完备搜索认证 ====================
print()
print("=== 阶段 3：分支定界完备搜索（对有限问题给出最优性认证） ===")


def exhaustive_max_sidon(n):
    """分支定界（branch and bound）。平移不变性允许 WLOG 含 0；
    剪枝判据：计数上界 kmax 与剩余空间估计。"""
    kmax = counting_max(n)
    best_size = 0
    best_set = []

    def rec(s, diffs, start):
        nonlocal best_size, best_set
        if len(s) > best_size:
            best_size = len(s)
            best_set = s[:]
        room = min(n - start, kmax - len(s))
        if len(s) + room <= best_size:
            return
        for x in range(start, n):
            new_diffs = [x - y for y in s]   # s 严格递增，此处 x 大于全部已有点
            if set(new_diffs) & diffs:
                continue
            s.append(x)
            diffs.update(new_diffs)
            rec(s, diffs, x + 1)
            s.pop()
            for nd in new_diffs:
                diffs.remove(nd)
            room = min(n - (x + 1), kmax - len(s))
            if len(s) + room <= best_size:
                return

    rec([0], set(), 1)
    return best_size, best_set


max_size, witness = exhaustive_max_sidon(N)
print("N=%d 上最大 Sidon 集规模 = %d（一个最优集：%s）" % (N, max_size, witness))
if max_size == len(elite):
    print("结论：迭代搜索命中的规模与完备搜索一致，即对该有限问题，")
    print("      闭环给出的是经过机械认证的最优解，而非仅是启发式好解。")
else:
    print("结论：迭代搜索未达最优（精英规模 %d < 最优 %d），提示提议分布或预算不足。"
          % (len(elite), max_size))

=== 阶段 0：验证器自检 ===
is_sidon([0, 1, 4, 10, 12, 25]) = True
is_sidon([0, 1, 2, 4, 10, 20]) = False 冲突：差 1 重复，出现于 (0, 1) 与 (1, 2)
计数上界（C(k,2) <= N-1）：k_max = 8

=== 阶段 1：无引导提议（均匀提议分布，每个目标规模 B=2000 次） ===
k=4：提议 2000 次，通过 1511 次，经验命中率 p_hat = 0.7555
k=5：提议 2000 次，通过 753 次，经验命中率 p_hat = 0.3765
k=6：提议 2000 次，通过 134 次，经验命中率 p_hat = 0.0670
k=7：提议 2000 次，通过 1 次，经验命中率 p_hat = 0.0005
被拒绝候选示例（验证器拦截）：
  k=4 候选 [6, 14, 17, 20]：差 3 重复，出现于 (14, 17) 与 (17, 20)
  k=5 候选 [23, 25, 27, 28, 29]：差 2 重复，出现于 (23, 25) 与 (25, 27)
  k=6 候选 [4, 12, 14, 19, 20, 30]：差 8 重复，出现于 (4, 12) 与 (12, 20)
  k=7 候选 [4, 6, 14, 20, 22, 24, 26]：差 16 重复，出现于 (4, 20) 与 (6, 22)

=== 阶段 2：迭代搜索（贪心构造 + 变异 + 精英保留，目标规模 7） ===
精英更新轨迹（提议序号, 精英规模, 精英集合）：
  第    1 次提议：规模 6，精英 = [3, 4, 6, 15, 23, 30]
  第   15 次提议：规模 7，精英 = [3, 4, 6, 11, 15, 25, 31]
总计提议 15 次；验证器拒绝 0 次；通过验证但未优于精英 13 次（保留规则拒绝）
最终精英（规模 7）：[3, 4, 6, 11, 15, 25, 31]

=== 阶段 3：分支定界完备搜索（对有限问题给出最优性认证） ===
N=32 上最大 Sidon 集规模 = 7（一个最优集：[0, 1, 3, 7, 12, 20, 30]）
结论：迭代搜索命中的规模与完备搜索一致，即对该有限